In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import pandas as pd
import openai


In [ ]:
qdrant_client = QdrantClient(
    url="http://localhost:6333"
)

qdrant_client.create_collection(
    collection_name="Amazon-items-collections-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

In [3]:
df_items = pd.read_json("../data/meta_Electronics_2022_2023_with_main_category_ratings_100_sample_1000.jsonl", lines=True)


In [4]:
df_items.head(2)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN


In [5]:
def preprocessed_data(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [6]:
df_items["preprocessed_data"] = df_items.apply(preprocessed_data, axis=1)

In [7]:
df_sample = df_items.sample(n=50, random_state=42)

In [8]:
def get_embeddings(text, model_name="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model_name
    )
    return response.data[0].embedding

In [9]:
from openai import embeddings


get_embeddings("hi, my name is Maheshwar")

data_to_embed = df_sample["preprocessed_data"].tolist()
pointstructs = []
for i, data in enumerate(data_to_embed):
    embeddings = get_embeddings(data)
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embeddings, 
            payload={"text": data}
        )
    )



In [10]:
pointstructs

[PointStruct(id=0, vector=[0.010930042713880539, -0.015056160278618336, 0.01137808058410883, -0.011284304782748222, -0.0503886416554451, -0.0004936232580803335, -0.03859378397464752, 0.04171963036060333, 0.002547564683482051, -0.010346551425755024, -0.0291745662689209, 0.014462249353528023, -0.03096671774983406, 0.08677349239587784, 0.01424343977123499, 0.004819013178348541, -0.03596807271242142, 0.009127471596002579, -0.018807174637913704, 0.016889989376068115, 0.01028924435377121, 0.0174943208694458, 0.033800818026065826, 0.03059161640703678, 0.021776730194687843, 0.02110988274216652, -0.03313397243618965, -0.026548855006694794, -0.023027068004012108, 0.05222247168421745, -0.013222330249845982, -0.020338840782642365, -0.01941150613129139, -0.049179982393980026, -0.04113613814115524, -0.019265633076429367, -0.017202574759721756, 0.01579594425857067, -0.018254943192005157, 0.0005453950725495815, 0.024214889854192734, 0.04019838199019432, 0.00795527920126915, 0.0011038144584745169, -0.0

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collections-00",
    wait=True,
    points=pointstructs
)

In [31]:
def retrieve_data(query):
    query_embeddings = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collections-00",
        query=query_embeddings
    )
    return results

In [32]:
retrieve_data("What airphones can I get?").points

[ScoredPoint(id=12, version=0, score=0.4969469, payload={'text': "TELSOR Wireless Earbuds for iPhone, Bluetooth Headphones Touch Control Stereo Sound Bluetooth Earbuds with Noise Cancelling Mic for Calls, 30H Playtime, IPX7 Waterproof Earbuds for Android, Black ♬【Bluetooth】Pair instantly with an uninterrupted and stable transmission with Bluetooth 5.1. AVRCP, HCP, HSP, and A2DP profiles are supported. The wireless earbuds are compatible with most Bluetooth enabled iPhones, Andriods, smart TVs, computers, etc. Each wireless earbuds will pair with each other when they are removed from the charging case. From here, enable Bluetooth on your chosen device and pair with the headphones. ♬【Clear Call & Sound quality】Each wireless earbuds features a 10mm diameter speaker and 2 microphones to reduce ambient noise and transmit your voice for a clear call in any environment. For the music enjoyers, these wireless earbuds offer a deep bass for an immersive musical experience. ♬【Long Battery Life & 

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

qdrant_client.create_collection(
    collection_name="Amazon-items-collections-02",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

In [11]:
df_sample = df_items.sample(n=50, random_state=25)

In [33]:
def get_embeddings(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [13]:

data_to_embed = df_sample["preprocessed_data"].tolist()
pointstructs = []
for i, data in enumerate(data_to_embed):
    embeddings = get_embeddings(data)
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embeddings, 
            payload={"text": data}
        )
    )


In [34]:
qdrant_client.upsert(
    collection_name="Amazon-items-collections-02",
    wait=True,
    points=pointstructs
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [14]:
import json

output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "integer",
                    "description": "Index of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the contexr.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application and I have a collection of 50 chinks of text.
The RAG Application will act as a shopping assistant that can answer questions about the stock of the product we have available.
I will provide all of the available products to you with indexes of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
As an output I need you to provide me with a list of questions and the index of the chunk that contains the answer to the question.
Also, provide an example answer to the questions given the context of the chunk.
Also, provide a reason why you chose the chunks to answer the question.
Try to have a mix of questions that use multiple chunks and questions that use a single chunk.
Also, include 5 questions that cannot be answered with the available chunks.

<OUTPUT JSON FORMAT>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON FORMAT>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text.
{[{"id": i, "text": data} for i, data in enumerate(data_to_embed)]}
"""


In [15]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text.
[{'id': 0, 'text': 'Bluetooth Car Adapter, LDNIO Bluetooth FM Transmitter for Car, 43W PD&QC 3.0 Three USB Port Car Bluetooth Adapter with LED Display, Hands-Free Calling, and AUX Input for All Smartphones Audio Player Fast Charging Type C Multi Ports: USB Type C Durable Fast Connect & Clear Bluetooth Sound 3 in 1 Value'}, {'id': 1, 'text': 'Bluetooth Multi-Device Keyboard, Dual Channel Universal Rechargeable Wireless Keyboard with Integrated Stand for iPad Smartphone Tablet MacBook iOS Windows Android Devices - Pink 【Easy-Switch to 2 Devices】 Simply press the FN+BT1/BT2 to switch typing between 2 connected Bluetooth devices, work with your smartphone and tablet on the slot stablely. 【Type Anywhere in Comfort】0.46in thick and 11.34in long, the compact Bluetooth keyboard is small enough to tuck into your briefcase and light enough to hold in hand. Minimalist layout lets you multitask at home or on the go. 【

In [16]:
response = openai.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
)

print(response.choices[0].message.content)

```json
[
  {
    "question": "Do you sell any Bluetooth car adapters that allow hands-free calling?",
    "chunk_ids": [0],
    "answer_example": "Yes, we have the LDNIO Bluetooth FM Transmitter for Car, which supports hands-free calling, fast charging, and multiple USB ports.",
    "reasoning": "Chunk 0 specifically describes a Bluetooth car adapter with hands-free calling."
  },
  {
    "question": "Which keyboards in your inventory can connect to more than one device at once?",
    "chunk_ids": [1, 25, 28],
    "answer_example": "We stock several multi-device keyboards, like the Bluetooth Multi-Device Keyboard (chunk 1), the Nasuque Bluetooth Keyboard for Mac (chunk 25), and the Samsers Ultra Slim Keyboard and Mouse Combo (chunk 28).",
    "reasoning": "Chunks 1, 25, and 28 each list features about connecting to multiple devices."
  },
  {
    "question": "Do you have any wireless earbuds with active noise cancelling?",
    "chunk_ids": [2],
    "answer_example": "Yes. The Active N

In [17]:
import json

json_output = response.choices[0].message.content
if json_output is not None:
    json_output = json_output.replace("```json", "")
    json_output = json_output.replace("```", "")
    json_output = json_output.replace("// Cannot be answered with available chunks (CHUNKS MISSING REQUIRED CONTEXT)", "")
    json_output = json.loads(json_output)
else:
    raise ValueError("response.choices[0].message.content is None")

In [21]:
json_output

[{'question': 'Do you sell any Bluetooth car adapters that allow hands-free calling?',
  'chunk_ids': [0],
  'answer_example': 'Yes, we have the LDNIO Bluetooth FM Transmitter for Car, which supports hands-free calling, fast charging, and multiple USB ports.',
  'reasoning': 'Chunk 0 specifically describes a Bluetooth car adapter with hands-free calling.'},
 {'question': 'Which keyboards in your inventory can connect to more than one device at once?',
  'chunk_ids': [1, 25, 28],
  'answer_example': 'We stock several multi-device keyboards, like the Bluetooth Multi-Device Keyboard (chunk 1), the Nasuque Bluetooth Keyboard for Mac (chunk 25), and the Samsers Ultra Slim Keyboard and Mouse Combo (chunk 28).',
  'reasoning': 'Chunks 1, 25, and 28 each list features about connecting to multiple devices.'},
 {'question': 'Do you have any wireless earbuds with active noise cancelling?',
  'chunk_ids': [2],
  'answer_example': 'Yes. The Active Noise Cancelling Wireless Earbuds come with ANC tec

In [36]:
from langsmith import Client
import os

client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
dataset_name = "rag-evaluation-dataset"
# dataset = client.create_dataset(
#     dataset_name=dataset_name,
#     description="Dataset for RAG evaluation"
# )

In [25]:
json_output

[{'question': 'Do you sell any Bluetooth car adapters that allow hands-free calling?',
  'chunk_ids': [0],
  'answer_example': 'Yes, we have the LDNIO Bluetooth FM Transmitter for Car, which supports hands-free calling, fast charging, and multiple USB ports.',
  'reasoning': 'Chunk 0 specifically describes a Bluetooth car adapter with hands-free calling.'},
 {'question': 'Which keyboards in your inventory can connect to more than one device at once?',
  'chunk_ids': [1, 25, 28],
  'answer_example': 'We stock several multi-device keyboards, like the Bluetooth Multi-Device Keyboard (chunk 1), the Nasuque Bluetooth Keyboard for Mac (chunk 25), and the Samsers Ultra Slim Keyboard and Mouse Combo (chunk 28).',
  'reasoning': 'Chunks 1, 25, and 28 each list features about connecting to multiple devices.'},
 {'question': 'Do you have any wireless earbuds with active noise cancelling?',
  'chunk_ids': [2],
  'answer_example': 'Yes. The Active Noise Cancelling Wireless Earbuds come with ANC tec

In [38]:
import json

if json_output is None:
    raise ValueError("json_output is None")

for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "context_ids": item["chunk_ids"],
            "contexts": [qdrant_client.retrieve(collection_name="Amazon-items-collections-02", ids=[id], with_payload=True)[0].payload["text"] for id in item["chunk_ids"]]
        }
    )